# 02 - KPI baseline

Notebook tinh baseline theo dinh nghia KPI canonical trong SPEC.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
EXPORT_DIR = ROOT / 'dashboard' / 'exports'
PROCESSED_DIR = ROOT / 'data' / 'processed'

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 140)


def read_csv(relative_path):
    path = ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(f'Missing required input: {path}')
    return pd.read_csv(path)


def pct(numerator, denominator):
    return np.where(pd.Series(denominator).replace(0, np.nan).notna(), numerator / denominator * 100, np.nan)

posts = read_csv('data/processed/posts.csv')
comments = read_csv('data/processed/comments.csv')
executive = read_csv('dashboard/exports/vw_executive_overview.csv')
daily = read_csv('dashboard/exports/vw_daily_engagement.csv')


In [ ]:
posts_kpi = posts.assign(
    engagement_count=posts['likes'] + posts['comments_count'] + posts['shares'],
    engagement_rate_calc=np.where(posts['reach'] > 0, (posts['likes'] + posts['comments_count'] + posts['shares']) / posts['reach'] * 100, np.nan),
    virality_score_calc=np.where(posts['reach'] > 0, posts['shares'] / posts['reach'] * 100, np.nan),
)
baseline = pd.DataFrame([{
    'date_from': posts['date'].min(),
    'date_to': posts['date'].max(),
    'total_posts': len(posts),
    'total_reach': posts['reach'].sum(),
    'total_impressions': posts['impressions'].sum(),
    'total_engagement': posts_kpi['engagement_count'].sum(),
    'avg_engagement_rate': posts_kpi['engagement_rate_calc'].mean(),
    'avg_virality_score': posts_kpi['virality_score_calc'].mean(),
    'sentiment_ratio': (comments['sentiment_label'].eq('positive').sum() / len(comments) * 100) if len(comments) else np.nan,
}])
display(baseline.round(4))
display(executive)


In [ ]:
daily_sorted = daily.assign(full_date=pd.to_datetime(daily['full_date'])).sort_values('full_date')
daily_sorted['previous_reach'] = daily_sorted['total_reach'].shift(1)
daily_sorted['reach_growth'] = np.where(daily_sorted['previous_reach'] > 0, (daily_sorted['total_reach'] - daily_sorted['previous_reach']) / daily_sorted['previous_reach'] * 100, np.nan)
display(daily_sorted[['full_date', 'post_count', 'total_reach', 'total_engagement', 'avg_engagement_rate', 'reach_growth']].tail(10))


In [ ]:
page_baseline = posts_kpi.groupby('page_name', as_index=False).agg(
    post_count=('post_id', 'count'),
    total_reach=('reach', 'sum'),
    total_engagement=('engagement_count', 'sum'),
    avg_engagement_rate=('engagement_rate_calc', 'mean'),
    avg_virality_score=('virality_score_calc', 'mean'),
).sort_values(['total_reach', 'total_engagement'], ascending=False)
display(page_baseline.head(10).round(4))


## Insight

- Key finding: Baseline YouTube hien tai co do phu lon nhung engagement rate trung binh con thap.
- Supporting data: 65 posts, 6.24M reach, 1.15K engagements, avg engagement rate export 0.744%.
- Recommended action: Dung baseline nay lam moc truoc khi toi uu noi dung va so sanh theo tung lan refresh.